# B2B CPRScore — Cohort Methodology Audit & Stress Test

This notebook traces **every step** of the cohort-based CPRScore calculation for a specific client
and validates the result against the consolidated report output.

## What this notebook shows

| Section | Description | Business Question Answered |
|---------|-------------|----------------------------|
| **A** | Raw loan population in B2BPricingHist | Q1: How many loans? Q2: How many have Cohort_ID? |
| **B** | Cohort distribution for the client | Q3: Unique cohorts? Q4: Max/min loans per cohort? |
| **C** | Full cohort detail table | Q5: Per-cohort ActualCPR, ModelCPR, client vs. total loans, other clients |
| **D** | Step-by-step CPRScore derivation — **single month** | Show the formula, each number, the final score |
| **E** | CPRScore as used in the report — **all history** | Matches the number in B2B_cohort_score_202601.xlsx |
| **F** | Cross-validation against Excel | Confirm ActualCPR, ModelCPR, CPRScore, RecapScore all reconcile |

> **Note on two calculation windows:** Section D computes CPRScore for the single specified month only.  
> Section E replicates the exact multi-month calculation used in the consolidated report, which spans all
> available history in `B2BPricingHist` (Feb 2023 – Jan 2026 in this run). Section E is the number you see in the Excel.

## 0. User Inputs — Change These to Audit Any Client / Month

In [1]:
from datetime import date

# ── Who and when to audit ────────────────────────────────────────────────────
CLIENT_NAME   = "LENNAR MORTGAGE LLC"            # Client name exactly as it appears in B2BPricingHist
ASOFDT        = date(2026, 1, 31)           # Month to audit

# ── Report output for cross-validation ──────────────────────────────────────
REPORT_EXCEL  = r"E:\Development\Shashi\B2B-Pricing-Model\Cohort_Analysis\B2B_reporting_raw\B2B_cohort_score_20260131.xlsx"
REPORT_SHEET  = 'rawTotal'                  # Sheet containing the AllMetrics data

# ── Minimum loan count threshold (must match functions file) ─────────────────
MIN_COHORT_LOANS = 10

print(f"Auditing: {CLIENT_NAME}  |  AsofDt: {ASOFDT}")

Auditing: LENNAR MORTGAGE LLC  |  AsofDt: 2026-01-31


## 1. Imports, Connection, and Reporter

In [2]:
import os, sys
import datetime
import numpy as np
import pandas as pd
from sqlalchemy import text

# Load the functions library
FUNCTIONS_DIR = os.path.join(os.getcwd(), 'functions')
if FUNCTIONS_DIR not in sys.path:
    sys.path.insert(0, FUNCTIONS_DIR)

%load_ext autoreload
%autoreload 2

from B2B_functions_cohort_score import sql_connection, B2BPricingReporter

engine = sql_connection()
print('Engine:', engine.url)

Engine: mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DVMC2VMDBRADC01P%3BDATABASE%3DAnalytics_Rpt%3BTrusted_Connection%3Dyes


In [3]:
# Load reporter — same parameters as the consolidated report run
# This populates reporter.perform_all (all history) and reporter.volume_all
reporter = B2BPricingReporter(engine).load(report_as_of_dt=ASOFDT)
print()
print(f"perform_all: {reporter.perform_all.shape[0]:,} rows across {reporter.perform_all['AsofDt'].nunique()} months")

Report as of  : 2026-01-31
Funding window: 2025-02-01  →  2026-01-31
Score window  : 1900-01-01  →  2026-01-31

perform_all: 8,266,965 rows across 36 months


## Section A — Raw Loan Population  (Q1 & Q2)

Answers:
- **Q1**: How many LoanNums does this client have in `B2BPricingHist` for the specified month?
- **Q2**: Of those, how many are considered for CPRScore (i.e., have a `Cohort_ID`)?

In [4]:
sql_loans = text("""
    SELECT
        h.LoanNum,
        h.BeginPrinBal,
        h.EndPrinBal,
        h.ProductCategory,
        h.DelqStatusMBA,
        h.PIFReason,
        h.PIFBal,
        ISNULL(c.Cohort_unique_ID, 'NO COHORT (PLS/unmapped)') AS Cohort_Status,
        c.Cohort_unique_ID as Cohort_ID,
        c.InvestorCatgy,
        c.LoanAge,
        c.Rule_Bucket,
        c.FICOOrig,
        c.IntRate,
        c.Market_Rate
    FROM Analytics_Rpt.md.B2BPricingHist h
    LEFT JOIN Analytics_Rpt.md.B2B_revamp_cohorts c
           ON h.LoanNum = c.LoanNum
          AND h.AsofDt  = c.AsofDt
    WHERE h.Client       = :client
      AND h.AsofDt       = :asofdt
      AND h.FundedLoanAmt > 0
    ORDER BY c.Cohort_ID, h.LoanNum
""")

df_loans = pd.read_sql(sql_loans, engine, params={'client': CLIENT_NAME, 'asofdt': ASOFDT})

n_total     = len(df_loans)
n_with_coh  = df_loans['Cohort_ID'].notna().sum()
n_no_coh    = n_total - n_with_coh

print(f"{'─'*60}")
print(f"Client  : {CLIENT_NAME}")
print(f"AsofDt  : {ASOFDT}")
print(f"{'─'*60}")
print(f"Q1. Total LoanNums in B2BPricingHist  : {n_total:>8,}")
print(f"Q2. LoanNums WITH  Cohort_ID (scored) : {n_with_coh:>8,}  ({100*n_with_coh/n_total:.1f}%)")
print(f"    LoanNums WITHOUT Cohort_ID (excl.) : {n_no_coh:>8,}  ({100*n_no_coh/n_total:.1f}%)")
print()
if n_no_coh > 0:
    print("Loans excluded from CPRScore (no Cohort_ID):")
    print(df_loans[df_loans['Cohort_ID'].isna()][['LoanNum','ProductCategory','InvestorCatgy','BeginPrinBal']]
          .to_string(index=False))

────────────────────────────────────────────────────────────
Client  : LENNAR MORTGAGE LLC
AsofDt  : 2026-01-31
────────────────────────────────────────────────────────────
Q1. Total LoanNums in B2BPricingHist  :    8,993
Q2. LoanNums WITH  Cohort_ID (scored) :    8,992  (100.0%)
    LoanNums WITHOUT Cohort_ID (excl.) :        1  (0.0%)

Loans excluded from CPRScore (no Cohort_ID):
   LoanNum ProductCategory InvestorCatgy  BeginPrinBal
7913515248    Conventional           PLS     368246.78


In [5]:
# Product-category breakdown of the scored population
print("Loan population breakdown by ProductCategory:")
print(
    df_loans.groupby('ProductCategory', dropna=False)
    .agg(LoanCount=('LoanNum','count'), TotalUPB=('BeginPrinBal','sum'))
    .assign(TotalUPB=lambda d: d['TotalUPB'].apply(lambda x: f'${x:,.0f}'))
    .to_string()
)

Loan population breakdown by ProductCategory:
                 LoanCount        TotalUPB
ProductCategory                           
Conventional          7859  $2,489,644,286
FHA                    874    $322,156,568
Jumbo                   17      $9,743,986
USDA                    37      $9,731,255
VA                     206     $82,851,932


## Section B — Cohort Distribution  (Q3 & Q4)

Answers:
- **Q3**: How many unique cohorts did the client's loans fall into?
- **Q4**: What was the max and min loan count across those cohorts?

In [6]:
df_scored = df_loans[df_loans['Cohort_ID'].notna()].copy()

coh_dist = (
    df_scored.groupby('Cohort_ID')
    .agg(
        LoanCount     = ('LoanNum',       'count'),
        TotalUPB      = ('BeginPrinBal',  'sum'),
        AvgBeginBal   = ('BeginPrinBal',  'mean'),
        InvestorCatgy = ('InvestorCatgy', 'first'),
    )
    .reset_index()
    .sort_values('LoanCount', ascending=False)
    .reset_index(drop=True)
)
coh_dist['Meets_Threshold'] = coh_dist['LoanCount'] >= MIN_COHORT_LOANS

n_cohorts      = len(coh_dist)
n_above_thresh = coh_dist['Meets_Threshold'].sum()
n_below_thresh = n_cohorts - n_above_thresh
max_row        = coh_dist.loc[coh_dist['LoanCount'].idxmax()]
min_row        = coh_dist.loc[coh_dist['LoanCount'].idxmin()]

print(f"Q3. Unique cohorts for {CLIENT_NAME} in {ASOFDT}: {n_cohorts}")
print()
print(f"Q4. Loan count per cohort:")
print(f"    Maximum : {int(max_row.LoanCount):>4}  (cohort {max_row.Cohort_ID})")
print(f"    Minimum : {int(min_row.LoanCount):>4}  (cohort {min_row.Cohort_ID})")
print(f"    Mean    : {coh_dist.LoanCount.mean():>5.1f}")
print()
print(f"    Cohorts meeting min-loan threshold (≥{MIN_COHORT_LOANS}) : {n_above_thresh}  → included in CPRScore")
print(f"    Cohorts BELOW threshold (<{MIN_COHORT_LOANS})             : {n_below_thresh}  → CPRScore defaults to 1.0 for these")
print()
print("All cohorts (sorted by loan count):")
display(coh_dist.style.apply(
    lambda col: ['background-color: #ffe8e8' if not v else '' for v in col],
    subset=['Meets_Threshold']))

Q3. Unique cohorts for LENNAR MORTGAGE LLC in 2026-01-31: 137

Q4. Loan count per cohort:
    Maximum : 1365  (cohort C079)
    Minimum :    1  (cohort C135)
    Mean    :  65.6

    Cohorts meeting min-loan threshold (≥10) : 94  → included in CPRScore
    Cohorts BELOW threshold (<10)             : 43  → CPRScore defaults to 1.0 for these

All cohorts (sorted by loan count):


,Cohort_ID,LoanCount,TotalUPB,AvgBeginBal,InvestorCatgy,Meets_Threshold
0,C079,1365,344185320.410000,252150.417883,FNMA,True
1,C080,1111,300853903.460000,270795.592673,FNMA,True
2,C066,1078,292016352.090000,270887.154072,FNMA,True
3,C092,408,208527774.380000,511097.486225,FNMA,True
4,C075,384,197884111.890000,515323.208047,FNMA,True
5,C078,246,67739250.240000,275362.805854,FNMA,True
6,C086,228,27073918.440000,118745.256316,FNMA,True
7,C124,150,32088499.160000,213923.327733,GNMA,True
8,C065,148,40241597.090000,271902.683041,FNMA,True
9,C068,138,36600201.720000,265218.853043,FNMA,True


## Section C — Cohort Detail Table  (Q5)

For **each cohort the client falls into** (single-month view — Jan 2026 only), this table shows:

| Column | Meaning |
|--------|---------|
| `Client_LoanCt` | Number of this client's loans in this cohort |
| `Total_LoanCt` | Total loans in this cohort from ALL clients |
| `Num_Other_Clients` | Unique clients OTHER than the audited client in this cohort |
| `Client_ActualCPR` | This client's actual CPR in this cohort: `1-(EndBal/BeginBal)^12` |
| `ModelCPR_cohort` | Cohort benchmark: `Σ(ActualCPR_gc × LoanCt_gc) / Σ(LoanCt_gc)` over all clients in cohort |
| `CPR_Ratio` | Client_ActualCPR / ModelCPR_cohort (ratio > 1 = faster prepayment than benchmark) |

> **ModelCPR_cohort** in this section is computed from **January 2026 single-month data only**.
> Section E shows the same using all 36 months (which is what the report uses).

In [7]:
# ── Filter perform_all to the report month ───────────────────────────────────
_tgt = ASOFDT
pa_month = reporter.perform_all[
    reporter.perform_all['AsofDt'].apply(
        lambda x: (x.date() if hasattr(x, 'date') else x) == _tgt)
].copy()

print(f"perform_all rows for {_tgt}: {len(pa_month):,}")

# ── Step 1: Client cohort cells ──────────────────────────────────────────────
client_cells = (
    pa_month[
        (pa_month['Client'] == CLIENT_NAME) &
        (pa_month['Cohort_ID'].notna())
    ]
    .groupby('Cohort_ID', dropna=False)
    .agg(
        Client_LoanCt  = ('LoanCt',      'sum'),
        Client_BeginBal= ('BeginPrinBal', 'sum'),
        Client_EndBal  = ('EndPrinBal',   'sum'),
    )
    .reset_index()
)
client_cells['Client_ActualCPR'] = np.where(
    client_cells['Client_BeginBal'] > 0,
    1 - (client_cells['Client_EndBal'] / client_cells['Client_BeginBal']) ** 12,
    np.nan
)

# ── Step 2: ALL clients in the same cohorts ──────────────────────────────────
client_cohort_ids = client_cells['Cohort_ID'].tolist()
all_in_cohorts = (
    pa_month[
        pa_month['Cohort_ID'].isin(client_cohort_ids) &
        pa_month['Cohort_ID'].notna()
    ]
    .groupby(['Cohort_ID', 'Client'], dropna=False)
    .agg(
        LoanCt_gc  = ('LoanCt',      'sum'),
        BeginBal_gc= ('BeginPrinBal', 'sum'),
        EndBal_gc  = ('EndPrinBal',   'sum'),
    )
    .reset_index()
)
all_in_cohorts['ActualCPR_gc'] = np.where(
    all_in_cohorts['BeginBal_gc'] > 0,
    1 - (all_in_cohorts['EndBal_gc'] / all_in_cohorts['BeginBal_gc']) ** 12,
    np.nan
)

# ── Step 3: ModelCPR_cohort — pooled from all clients meeting the threshold ──
valid_cells = all_in_cohorts[
    (all_in_cohorts['LoanCt_gc'] >= MIN_COHORT_LOANS) &
    all_in_cohorts['ActualCPR_gc'].notna()
].copy()

model_cpr_single = (
    valid_cells.groupby('Cohort_ID', dropna=False)
    .apply(lambda g: pd.Series({
        'ModelCPR_cohort_1M':     (g['ActualCPR_gc'] * g['LoanCt_gc']).sum() / g['LoanCt_gc'].sum(),
        'Total_LoanCt_cohort':     g['LoanCt_gc'].sum(),
        'Num_Clients_cohort':      g['Client'].nunique(),
        'Num_OtherClients_cohort': g[g['Client'] != CLIENT_NAME]['Client'].nunique(),
    }))
    .reset_index()
)

# ── Step 4: Assemble the detail table ───────────────────────────────────────
detail = client_cells.merge(model_cpr_single, on='Cohort_ID', how='left')
detail['Meets_Threshold'] = detail['Client_LoanCt'] >= MIN_COHORT_LOANS
detail['CPR_Ratio_1M']    = np.where(
    (detail['ModelCPR_cohort_1M'] > 0) & detail['ModelCPR_cohort_1M'].notna(),
    detail['Client_ActualCPR'] / detail['ModelCPR_cohort_1M'],
    np.nan
)

display_cols = ['Cohort_ID', 'Client_LoanCt', 'Total_LoanCt_cohort', 'Num_OtherClients_cohort',
                'Client_ActualCPR', 'ModelCPR_cohort_1M', 'CPR_Ratio_1M', 'Meets_Threshold']
detail_display = detail[display_cols].copy()
for col in ['Client_ActualCPR', 'ModelCPR_cohort_1M', 'CPR_Ratio_1M']:
    detail_display[col] = detail_display[col].map(lambda x: f'{x:.6f}' if pd.notna(x) else 'N/A')

print(f"Q5. Cohort Detail Table for {CLIENT_NAME}  ({_tgt}, single-month):")
print(f"    {len(detail)} cohorts total, {detail['Meets_Threshold'].sum()} meet the ≥{MIN_COHORT_LOANS} loan threshold")
print()
display(detail_display)

perform_all rows for 2026-01-31: 300,638
Q5. Cohort Detail Table for LENNAR MORTGAGE LLC  (2026-01-31, single-month):
    137 cohorts total, 94 meet the ≥10 loan threshold



C:\Users\kanshash\AppData\Local\Temp\10\ipykernel_24224\17643062.py:59: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,Cohort_ID,Client_LoanCt,Total_LoanCt_cohort,Num_OtherClients_cohort,Client_ActualCPR,ModelCPR_cohort_1M,CPR_Ratio_1M,Meets_Threshold
0,C001,15,967.0,36.0,0.000000,0.004743,0.000000,True
1,C002,25,3588.0,111.0,0.000000,0.007065,0.000000,True
2,C003,29,444.0,15.0,0.016338,0.005803,2.815644,True
3,C004,16,467.0,19.0,0.000000,0.000000,N/A,True
4,C005,90,237.0,8.0,0.000000,0.000000,N/A,True
...,...,...,...,...,...,...,...,...
132,C156,39,1095.0,35.0,0.000000,0.000913,0.000000,True
133,C157,79,437.0,7.0,0.000000,0.000741,0.000000,True
134,C158,11,46.0,3.0,0.000000,0.000000,N/A,True
135,C159,39,79.0,3.0,0.000000,0.000000,N/A,True


## Section D — Step-by-Step CPRScore Derivation  (Single Month)

Shows the exact arithmetic used to arrive at CPRScore for this client.

**Formula recap:**
$$\text{ActualCPR}_{\text{group}} = \frac{\sum (\text{ActualCPR}_{gc} \times \text{LoanCt}_{gc})}{\sum \text{LoanCt}_{gc}}$$

$$\text{ModelCPR}_{\text{group}} = \frac{\sum (\text{ModelCPR}_{\text{cohort}} \times \text{LoanCt}_{gc})}{\sum \text{LoanCt}_{gc}}$$

$$\text{CPRScore} = \frac{\text{ActualCPR}_{\text{group}}}{\text{ModelCPR}_{\text{group}}} = \frac{\sum (\text{ActualCPR}_{gc} \times \text{LoanCt}_{gc})}{\sum (\text{ModelCPR}_{\text{cohort}} \times \text{LoanCt}_{gc})}$$

Only cohorts where `Client_LoanCt >= 10` are included (others default CPRScore contribution to 1.0).

In [8]:
# ── Filter to valid cohorts (meet threshold + have ModelCPR) ─────────────────
valid = detail[
    detail['Meets_Threshold'] &
    detail['Client_ActualCPR'].notna() &
    detail['ModelCPR_cohort_1M'].notna()
].copy()

valid['ActualCPR_numerator']   = valid['Client_ActualCPR']   * valid['Client_LoanCt']
valid['ModelCPR_denominator']  = valid['ModelCPR_cohort_1M'] * valid['Client_LoanCt']

print(f"{'─'*72}")
print(f"STEP-BY-STEP CPRScore derivation for {CLIENT_NAME}  ({_tgt})")
print(f"{'─'*72}")
print()

# Show per-cohort contribution
step_table = valid[['Cohort_ID', 'Client_LoanCt', 'Client_ActualCPR',
                     'ModelCPR_cohort_1M', 'ActualCPR_numerator', 'ModelCPR_denominator']].copy()
for col in ['Client_ActualCPR', 'ModelCPR_cohort_1M', 'ActualCPR_numerator', 'ModelCPR_denominator']:
    step_table[col] = step_table[col].round(8)
display(step_table)
print()

sum_loanct      = valid['Client_LoanCt'].sum()
sum_actual_num  = valid['ActualCPR_numerator'].sum()
sum_model_den   = valid['ModelCPR_denominator'].sum()
actual_cpr_1M   = sum_actual_num / sum_loanct   if sum_loanct > 0 else np.nan
model_cpr_1M    = sum_model_den  / sum_loanct   if sum_loanct > 0 else np.nan
cpr_score_1M    = (actual_cpr_1M / model_cpr_1M) if (pd.notna(model_cpr_1M) and model_cpr_1M > 0) else 1.0

print(f"Σ LoanCt (valid cohorts)             = {sum_loanct:,}")
print(f"Σ (ActualCPR_gc × LoanCt_gc)         = {sum_actual_num:.8f}")
print(f"Σ (ModelCPR_cohort × LoanCt_gc)      = {sum_model_den:.8f}")
print()
print(f"ActualCPR  (single-month) = {sum_actual_num:.8f} / {sum_loanct} = {actual_cpr_1M:.8f}")
print(f"ModelCPR   (single-month) = {sum_model_den:.8f}  / {sum_loanct} = {model_cpr_1M:.8f}")
print(f"CPRScore   (single-month) = {actual_cpr_1M:.8f} / {model_cpr_1M:.8f} = {cpr_score_1M:.8f}")
print()
print(f"⚠  This is the single-month ({_tgt}) score.")
print(f"   Section E shows the ALL-HISTORY score which is what appears in the consolidated Excel.")

────────────────────────────────────────────────────────────────────────
STEP-BY-STEP CPRScore derivation for LENNAR MORTGAGE LLC  (2026-01-31)
────────────────────────────────────────────────────────────────────────



,Cohort_ID,Client_LoanCt,Client_ActualCPR,ModelCPR_cohort_1M,ActualCPR_numerator,ModelCPR_denominator
0,C001,15,0.000000,0.004743,0.0000,0.071144
1,C002,25,0.000000,0.007065,0.0000,0.176617
2,C003,29,0.016338,0.005803,0.4738,0.168274
3,C004,16,0.000000,0.000000,0.0000,0.000000
4,C005,90,0.000000,0.000000,0.0000,0.000000
...,...,...,...,...,...,...
132,C156,39,0.000000,0.000913,0.0000,0.035612
133,C157,79,0.000000,0.000741,0.0000,0.058551
134,C158,11,0.000000,0.000000,0.0000,0.000000
135,C159,39,0.000000,0.000000,0.0000,0.000000



Σ LoanCt (valid cohorts)             = 8,834
Σ (ActualCPR_gc × LoanCt_gc)         = 293.84935107
Σ (ModelCPR_cohort × LoanCt_gc)      = 284.23216493

ActualCPR  (single-month) = 293.84935107 / 8834 = 0.03326345
ModelCPR   (single-month) = 284.23216493  / 8834 = 0.03217480
CPRScore   (single-month) = 0.03326345 / 0.03217480 = 1.03383567

⚠  This is the single-month (2026-01-31) score.
   Section E shows the ALL-HISTORY score which is what appears in the consolidated Excel.


## Section E — Full Report CPRScore  (All History, Matches Excel)

The consolidated report (`B2B_cohort_score_202601.xlsx`) uses **all available history** in
`B2BPricingHist` (Feb 2023 – Jan 2026, i.e., 36 months).  This section replicates
that exact multi-month calculation to produce the number you see in the `Client` metric row of the report.

The same formula applies, but `LoanCt_gc` and the balances now accumulate across all months.

In [9]:
# ── Use the full perform_all (all 36 months) — same data as the report run ───
pa_full = reporter.perform_all.copy()

# ── Client's cohort cells across ALL months ──────────────────────────────────
client_cells_full = (
    pa_full[
        (pa_full['Client'] == CLIENT_NAME) &
        (pa_full['Cohort_ID'].notna())
    ]
    .groupby('Cohort_ID', dropna=False)
    .agg(
        Client_LoanCt  = ('LoanCt',      'sum'),
        Client_BeginBal= ('BeginPrinBal', 'sum'),
        Client_EndBal  = ('EndPrinBal',   'sum'),
    )
    .reset_index()
)
client_cells_full['Client_ActualCPR'] = np.where(
    client_cells_full['Client_BeginBal'] > 0,
    1 - (client_cells_full['Client_EndBal'] / client_cells_full['Client_BeginBal']) ** 12,
    np.nan
)

# ── ALL clients in the same cohorts across ALL months ───────────────────────
cohort_ids_full = client_cells_full['Cohort_ID'].tolist()
all_in_cohorts_full = (
    pa_full[
        pa_full['Cohort_ID'].isin(cohort_ids_full) &
        pa_full['Cohort_ID'].notna()
    ]
    .groupby(['Cohort_ID', 'Client'], dropna=False)
    .agg(
        LoanCt_gc  = ('LoanCt',      'sum'),
        BeginBal_gc= ('BeginPrinBal', 'sum'),
        EndBal_gc  = ('EndPrinBal',   'sum'),
    )
    .reset_index()
)
all_in_cohorts_full['ActualCPR_gc'] = np.where(
    all_in_cohorts_full['BeginBal_gc'] > 0,
    1 - (all_in_cohorts_full['EndBal_gc'] / all_in_cohorts_full['BeginBal_gc']) ** 12,
    np.nan
)

# ── ModelCPR_cohort from all clients (meeting min-loans threshold) ────────────
valid_full = all_in_cohorts_full[
    (all_in_cohorts_full['LoanCt_gc'] >= MIN_COHORT_LOANS) &
    all_in_cohorts_full['ActualCPR_gc'].notna()
].copy()

model_cpr_full = (
    valid_full.groupby('Cohort_ID', dropna=False)
    .apply(lambda g: pd.Series({
        'ModelCPR_cohort': (g['ActualCPR_gc'] * g['LoanCt_gc']).sum() / g['LoanCt_gc'].sum(),
        'Total_LoanCt_cohort_allHistory': g['LoanCt_gc'].sum(),
        'Num_Clients_cohort_allHistory':  g['Client'].nunique(),
    }))
    .reset_index()
)

# ── Merge ModelCPR back to client cells ──────────────────────────────────────
detail_full = client_cells_full.merge(model_cpr_full, on='Cohort_ID', how='left')
detail_full['Meets_Threshold'] = detail_full['Client_LoanCt'] >= MIN_COHORT_LOANS
detail_full['CPR_Ratio_full']  = np.where(
    (detail_full['ModelCPR_cohort'] > 0) & detail_full['ModelCPR_cohort'].notna(),
    detail_full['Client_ActualCPR'] / detail_full['ModelCPR_cohort'],
    np.nan
)

# ── Final CPRScore ────────────────────────────────────────────────────────────
valid_final = detail_full[
    detail_full['Meets_Threshold'] &
    detail_full['Client_ActualCPR'].notna() &
    detail_full['ModelCPR_cohort'].notna()
].copy()

valid_final['Act_x_N']   = valid_final['Client_ActualCPR'] * valid_final['Client_LoanCt']
valid_final['Model_x_N'] = valid_final['ModelCPR_cohort']  * valid_final['Client_LoanCt']

sum_N_full      = valid_final['Client_LoanCt'].sum()
sum_act_full    = valid_final['Act_x_N'].sum()
sum_model_full  = valid_final['Model_x_N'].sum()
actual_cpr_full = sum_act_full   / sum_N_full  if sum_N_full > 0 else np.nan
model_cpr_full_val  = sum_model_full / sum_N_full  if sum_N_full > 0 else np.nan
cpr_score_full  = (actual_cpr_full / model_cpr_full_val) if (pd.notna(model_cpr_full_val) and model_cpr_full_val > 0) else 1.0

print(f"{'─'*72}")
print(f"FULL REPORT CPRScore for {CLIENT_NAME}  (all history: {pa_full['AsofDt'].nunique()} months)")
print(f"{'─'*72}")
print(f"Cohorts considered (all months)    : {len(detail_full)}")
print(f"Cohorts meeting threshold (≥{MIN_COHORT_LOANS})     : {valid_final['Cohort_ID'].nunique()}")
print(f"Loan-months in valid cohorts       : {sum_N_full:,}")
print()
print(f"Σ (ActualCPR_gc × LoanCt_gc)       = {sum_act_full:.8f}")
print(f"Σ (ModelCPR_cohort × LoanCt_gc)    = {sum_model_full:.8f}")
print()
print(f"ActualCPR  (all history) = {actual_cpr_full:.8f}")
print(f"ModelCPR   (all history) = {model_cpr_full_val:.8f}")
print(f"CPRScore   (all history) = {actual_cpr_full:.8f} / {model_cpr_full_val:.8f} = {cpr_score_full:.8f}")

────────────────────────────────────────────────────────────────────────
FULL REPORT CPRScore for LENNAR MORTGAGE LLC  (all history: 36 months)
────────────────────────────────────────────────────────────────────────
Cohorts considered (all months)    : 140
Cohorts meeting threshold (≥10)     : 127
Loan-months in valid cohorts       : 128,545

Σ (ActualCPR_gc × LoanCt_gc)       = 6232.68904706
Σ (ModelCPR_cohort × LoanCt_gc)    = 3560.17630474

ActualCPR  (all history) = 0.04848644
ModelCPR   (all history) = 0.02769595
CPRScore   (all history) = 0.04848644 / 0.02769595 = 1.75066865


C:\Users\kanshash\AppData\Local\Temp\10\ipykernel_24224\1819440064.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [10]:
# Per-cohort detail table for the ALL-HISTORY scoring  (helps trace each cohort's contribution)
print("Per-cohort contribution to the all-history CPRScore:")
show = detail_full[
    detail_full['Meets_Threshold']
].copy().sort_values('Client_LoanCt', ascending=False)
show['Act_x_N']   = (show['Client_ActualCPR'] * show['Client_LoanCt']).round(8)
show['Model_x_N'] = (show['ModelCPR_cohort']  * show['Client_LoanCt']).round(8)
show['CPR_Ratio'] = (show['Client_ActualCPR'] / show['ModelCPR_cohort']).round(6)
display_full_cols = ['Cohort_ID', 'Client_LoanCt', 'Total_LoanCt_cohort_allHistory',
                     'Num_Clients_cohort_allHistory',
                     'Client_ActualCPR', 'ModelCPR_cohort', 'CPR_Ratio', 'Act_x_N', 'Model_x_N']
display(show[display_full_cols].reset_index(drop=True))

Per-cohort contribution to the all-history CPRScore:


,Cohort_ID,Client_LoanCt,Total_LoanCt_cohort_allHistory,Num_Clients_cohort_allHistory,Client_ActualCPR,ModelCPR_cohort,CPR_Ratio,Act_x_N,Model_x_N
0,C080,16871,84801.0,492.0,0.012294,0.011767,1.044860,207.418106,198.512781
1,C092,11409,187133.0,524.0,0.003060,0.002828,1.081874,34.908383,32.266602
2,C066,9959,121742.0,548.0,0.004797,0.004757,1.008431,47.776176,47.376724
3,C082,8660,187947.0,423.0,0.010916,0.002915,3.744658,94.533644,25.244936
4,C079,8477,127636.0,606.0,0.010125,0.018822,0.537947,85.832780,159.556127
...,...,...,...,...,...,...,...,...,...
122,C070,14,11396.0,210.0,0.939466,0.297777,3.154928,13.152529,4.168884
123,C051,13,2143.0,96.0,0.000000,0.167784,0.000000,0.000000,2.181193
124,C133,13,27958.0,347.0,0.000000,0.004884,0.000000,0.000000,0.063498
125,C154,11,2814.0,79.0,0.999906,0.536671,1.863163,10.998970,5.903385


## Section F — Cross-Validation Against Consolidated Report Excel

Loads `B2B_cohort_score_202601.xlsx` and pulls the `MetricName = 'Client'` row for the audited client.
Every field should match what Section E computed.

In [11]:
# Load the Excel report
df_excel = pd.read_excel(REPORT_EXCEL, sheet_name=REPORT_SHEET)

# Find the client row (MetricName='Client', MetricValue=CLIENT_NAME, ProductCategory='All')
client_row = df_excel[
    (df_excel['MetricName'] == 'Client') &
    (df_excel['MetricValue'] == CLIENT_NAME) &
    (df_excel['ProductCategory'] == 'All')
]

if len(client_row) == 0:
    print(f"⚠  '{CLIENT_NAME}' not found in {REPORT_SHEET} sheet. Check CLIENT_NAME spelling.")
    print("Available client names (sample):")
    print(df_excel[df_excel['MetricName']=='Client']['MetricValue'].drop_duplicates().head(20).to_string())
else:
    row = client_row.iloc[0]
    print(f"{'═'*72}")
    print(f"RECONCILIATION: {CLIENT_NAME}  |  ProductCategory=All  |  {ASOFDT}")
    print(f"{'═'*72}")
    print()

    metrics = ['ActualCPR', 'ModelCPR', 'CPRScore', 'CPRLoanct']
    recon = pd.DataFrame({
        'Metric':         metrics,
        'From_Excel':     [row.get(m, 'N/A') for m in metrics],
        'From_SectionE':  [actual_cpr_full, model_cpr_full_val, cpr_score_full, sum_N_full],
        'From_SectionD':  [actual_cpr_1M, model_cpr_1M, cpr_score_1M, sum_loanct],
    })
    for col in ['From_Excel', 'From_SectionE', 'From_SectionD']:
        recon[col] = recon[col].apply(lambda x: f'{float(x):.8f}' if pd.notna(x) and x != 'N/A' else str(x))

    recon['Match_SectionE?'] = recon.apply(
        lambda r: '✅ MATCH' if abs(float(r.From_Excel) - float(r.From_SectionE)) < 1e-5
        else '❌ DIFF', axis=1)

    display(recon)
    print()

    # Full context row from Excel
    print("Full row from Excel (key columns):")
    key_cols = ['ProductCategory', 'MetricName', 'MetricValue',
                'LoanCt', 'FundedLoanAmt', 'CPRLoanct',
                'ActualCPR', 'ModelCPR', 'CPRScore',
                'CohortRecap', 'ActualRecap', 'RecapScore', 'TotalScore']
    display(client_row[key_cols])

════════════════════════════════════════════════════════════════════════
RECONCILIATION: LENNAR MORTGAGE LLC  |  ProductCategory=All  |  2026-01-31
════════════════════════════════════════════════════════════════════════



,Metric,From_Excel,From_SectionE,From_SectionD,Match_SectionE?
0,ActualCPR,0.04848600,0.04848644,0.03326345,✅ MATCH
1,ModelCPR,0.02769600,0.02769595,0.03217480,✅ MATCH
2,CPRScore,1.75066900,1.75066865,1.03383567,✅ MATCH
3,CPRLoanct,128638.00000000,128545.00000000,8834.00000000,❌ DIFF



Full row from Excel (key columns):


,ProductCategory,MetricName,MetricValue,LoanCt,FundedLoanAmt,CPRLoanct,ActualCPR,ModelCPR,CPRScore,CohortRecap,ActualRecap,RecapScore,TotalScore
122,All,Client,LENNAR MORTGAGE LLC,6362,1.927092e+09,128638.0,0.048486,0.027696,1.750669,0.101362,0.084792,0.836533,0.477836


## Section G — Summary

Comprehensive summary of all findings for the audited client.

In [12]:
print(f"{'═'*72}")
print(f"AUDIT SUMMARY: {CLIENT_NAME}  |  {ASOFDT}")
print(f"{'═'*72}")
print()
print(f"A. Loan Population ({ASOFDT} only):")
print(f"   Total loans in B2BPricingHist            : {n_total:,}")
print(f"   Loans with Cohort_ID (scored)            : {n_with_coh:,}  ({100*n_with_coh/n_total:.1f}%)")
print(f"   Loans excluded (no Cohort_ID)            : {n_no_coh:,}")
print()
print(f"B. Cohort Distribution ({ASOFDT} only):")
print(f"   Unique cohorts                           : {n_cohorts}")
print(f"   Cohorts meeting ≥{MIN_COHORT_LOANS} loan threshold      : {n_above_thresh}")
print(f"   Max loans in any single cohort           : {int(coh_dist.LoanCount.max())}")
print(f"   Min loans in any single cohort           : {int(coh_dist.LoanCount.min())}")
print()
print(f"C. CPRScore — Single Month ({ASOFDT}):")
print(f"   ActualCPR                                : {actual_cpr_1M:.6f}")
print(f"   ModelCPR (cohort benchmark)              : {model_cpr_1M:.6f}")
print(f"   CPRScore                                 : {cpr_score_1M:.6f}")
print()
print(f"D. CPRScore — All History (matches Excel):")
print(f"   Months of history used                   : {pa_full['AsofDt'].nunique()}")
print(f"   Loan-months in valid cohorts             : {sum_N_full:,}")
print(f"   ActualCPR                                : {actual_cpr_full:.6f}")
print(f"   ModelCPR (cohort benchmark)              : {model_cpr_full_val:.6f}")
print(f"   CPRScore                                 : {cpr_score_full:.6f}")
print()
print(f"E. Interpretation:")
if pd.notna(cpr_score_full):
    if cpr_score_full > 1.05:
        print(f"   CPRScore {cpr_score_full:.3f} > 1.0 → This client's loans prepay FASTER than")
        print(f"   their cohort peers ({(cpr_score_full-1)*100:.1f}% faster). Higher CPR = less favorable for MSR.")
    elif cpr_score_full < 0.95:
        print(f"   CPRScore {cpr_score_full:.3f} < 1.0 → This client's loans prepay SLOWER than")
        print(f"   their cohort peers ({(1-cpr_score_full)*100:.1f}% slower). Lower CPR = more favorable for MSR.")
    else:
        print(f"   CPRScore {cpr_score_full:.3f} ≈ 1.0 → This client's prepayment speed is roughly")
        print(f"   in line with their cohort peers.")

════════════════════════════════════════════════════════════════════════
AUDIT SUMMARY: LENNAR MORTGAGE LLC  |  2026-01-31
════════════════════════════════════════════════════════════════════════

A. Loan Population (2026-01-31 only):
   Total loans in B2BPricingHist            : 8,993
   Loans with Cohort_ID (scored)            : 8,992  (100.0%)
   Loans excluded (no Cohort_ID)            : 1

B. Cohort Distribution (2026-01-31 only):
   Unique cohorts                           : 137
   Cohorts meeting ≥10 loan threshold      : 94
   Max loans in any single cohort           : 1365
   Min loans in any single cohort           : 1

C. CPRScore — Single Month (2026-01-31):
   ActualCPR                                : 0.033263
   ModelCPR (cohort benchmark)              : 0.032175
   CPRScore                                 : 1.033836

D. CPRScore — All History (matches Excel):
   Months of history used                   : 36
   Loan-months in valid cohorts             : 128,545
   Actual